# 02 - Data Preprocessing
# Purpose

This notebook cleans and standardizes the raw Eurovision datasets to create a reproducible working dataset for downstream analysis. The preprocessing steps include data validation, duplicate removal, handling missing values, standardizing formats, and filtering the project scope to contests held from 2008 onwards.

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Import Project Configuration

import sys

sys.path.append("/content/drive/MyDrive/Eurovision_Prediction")

from config import *

import pandas as pd
import numpy as np

In [13]:
# Load Raw Data

contestants = pd.read_csv(RAW_DATA / "contestants.csv")
votes = pd.read_csv(RAW_DATA / "votes.csv")

print("Contestants:", contestants.shape)
print("Votes:", votes.shape)

Contestants: (2501, 21)
Votes: (56562, 9)


In [14]:
# Preview the Datasets

display(contestants.head())

display(votes.head())

,year,to_country_id,to_country,performer,song,place_contest,sf_num,running_final,running_sf,place_final,...,place_sf,points_sf,points_tele_final,points_jury_final,points_tele_sf,points_jury_sf,composers,lyricists,lyrics,youtube_url
0,1956,ch,Switzerland,Refrain,Lys Assia,1.0,NaN,9.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Géo Voumard,Émile Gardaz,"Refrain\n\n(Refrain d'amour…)\n\nRefrain, coul...",https://youtube.com/watch?v=9POvOONqAj0
1,1956,nl,Netherlands,Holland,De Vogels Van Jetty Paerl,NaN,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Cor Lemaire,Annie M. G. Schmidt,De vogels van Holland\n\nDe vogels van Holland...,https://youtube.com/watch?v=73FpzjAV15w
2,1956,ch,Switzerland,Lys Assia,Das Alte Karussell,NaN,NaN,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,George Betz-Stahl,NaN,Das alte Karussell\n\nDas alte Karussell\nDas ...,https://youtube.com/watch?v=lVpf8uzHnX0
3,1956,be,Belgium,Fud Leclerc,Messieurs Les Noyés De La Seine,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Jack Say;Jean Miret,Robert Montal,Messieurs les noyés de la Seine\n\nMessieurs l...,https://youtube.com/watch?v=U9O3sqlyra0
4,1956,de,Germany,Walter Andreas Schwarz,Im Wartesaal Zum Großen Glück,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Walter Andreas Schwarz,NaN,Im Wartesaal zum großen Glück\n\nEs gibt einen...,https://youtube.com/watch?v=jqo-whfb0mk


,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
0,1957,final,at,nl,at,nl,6,NaN,NaN
1,1957,final,at,fr,at,fr,0,NaN,NaN
2,1957,final,at,dk,at,dk,0,NaN,NaN
3,1957,final,at,lu,at,lu,3,NaN,NaN
4,1957,final,at,de,at,de,0,NaN,NaN


In [15]:
# Standardize Column Names

contestants.columns = (
    contestants.columns
    .str.strip()
    .str.lower()
)

votes.columns = (
    votes.columns
    .str.strip()
    .str.lower()
)

In [16]:
# Check Missing Values

print("Contestants Missing Values")
display(contestants.isna().sum())

print("Votes Missing Values")
display(votes.isna().sum())

Contestants Missing Values


,0
year,0
to_country_id,0
to_country,0
performer,0
song,2
place_contest,726
sf_num,1788
running_final,1027
running_sf,1788
place_final,1039


Votes Missing Values


,0
year,0
round,0
from_country_id,0
to_country_id,0
from_country,0
to_country,0
total_points,0
tele_points,39613
jury_points,39613


In [17]:
# Remove Duplicate Rows

contestants = contestants.drop_duplicates()
votes = votes.drop_duplicates()

print("Contestants:", contestants.shape)
print("Votes:", votes.shape)

Contestants: (2501, 21)
Votes: (56562, 9)


In [18]:
# Filter Project Scope

contestants = contestants[
    contestants["year"] >= 2008
].copy()

votes = votes[
    votes["year"] >= 2008
].copy()

print(contestants.shape)
print(votes.shape)

(1333, 21)
(30998, 9)


In [19]:
# Create a Unique Eurovision Entry dataset

contestants_unique = (
    contestants
    .drop_duplicates(
        subset=[
            "year",
            "to_country_id",
            "song",
            "performer"
        ]
    )
    .reset_index(drop=True)
)

print("Contestants (all performances):", len(contestants))
print("Unique Eurovision entries:", len(contestants_unique))

Contestants (all performances): 1333
Unique Eurovision entries: 718


In [20]:
# Compare Yearly Counts

print("All Performances")
display(
    contestants.groupby("year").size()
)

print()

print("Unique Songs")
display(
    contestants_unique.groupby("year").size()
)

All Performances


,0
year,
2008,81
2009,79
2010,73
2011,81
2012,78
2013,72
2014,68
2015,73
2016,78



Unique Songs


,0
year,
2008,43
2009,42
2010,39
2011,43
2012,42
2013,39
2014,37
2015,40
2016,42


In [21]:
#Check for Duplicate Unique Entries

duplicates = contestants_unique.duplicated(
    subset=[
        "year",
        "to_country_id",
        "song",
        "performer"
    ]
)

print("Duplicate unique entries:", duplicates.sum())

Duplicate unique entries: 0


In [22]:
# Inspect Data types

contestants_unique.info()

print()

votes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718 entries, 0 to 717
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               718 non-null    int64  
 1   to_country_id      718 non-null    object 
 2   to_country         718 non-null    object 
 3   performer          718 non-null    object 
 4   song               717 non-null    object 
 5   place_contest      717 non-null    float64
 6   sf_num             0 non-null      float64
 7   running_final      462 non-null    float64
 8   running_sf         0 non-null      float64
 9   place_final        462 non-null    float64
 10  points_final       462 non-null    float64
 11  place_sf           0 non-null      float64
 12  points_sf          0 non-null      float64
 13  points_tele_final  257 non-null    float64
 14  points_jury_final  257 non-null    float64
 15  points_tele_sf     0 non-null      float64
 16  points_jury_sf     0 non-n

In [23]:
# Basic Statistics

display(
    contestants_unique.describe(include="all")
)

display(
    votes.describe(include="all")
)

,year,to_country_id,to_country,performer,song,place_contest,sf_num,running_final,running_sf,place_final,...,place_sf,points_sf,points_tele_final,points_jury_final,points_tele_sf,points_jury_sf,composers,lyricists,lyrics,youtube_url
count,718.000000,718,718,718,717,717.000000,0.0,462.000000,0.0,462.000000,...,0.0,0.0,257.000000,257.000000,0.0,0.0,718,350,718,718
unique,NaN,48,48,579,715,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,685,335,718,718
top,NaN,gr,Greece,Love,Jedward,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,Dimitris Kontopoulos;Philipp Kirkorov,Karen Kavaleryan,Just Go\n\nI carried your love inside my heart...,https://youtube.com/watch?v=LQDvYwYIT54
freq,NaN,18,18,14,2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,5,4,1,1
mean,2016.610028,NaN,NaN,NaN,NaN,20.496513,NaN,13.385281,NaN,13.339827,...,NaN,NaN,89.595331,88.692607,NaN,NaN,NaN,NaN,NaN,NaN
std,5.527768,NaN,NaN,NaN,NaN,11.638601,NaN,7.436123,NaN,7.423017,...,NaN,NaN,94.103166,75.395020,NaN,NaN,NaN,NaN,NaN,NaN
min,2008.000000,NaN,NaN,NaN,NaN,1.000000,NaN,1.000000,NaN,1.000000,...,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,2012.000000,NaN,NaN,NaN,NaN,10.000000,NaN,7.000000,NaN,7.000000,...,NaN,NaN,18.000000,34.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,2016.000000,NaN,NaN,NaN,NaN,20.000000,NaN,13.000000,NaN,13.000000,...,NaN,NaN,53.000000,67.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,2022.000000,NaN,NaN,NaN,NaN,30.000000,NaN,20.000000,NaN,20.000000,...,NaN,NaN,136.000000,129.000000,NaN,NaN,NaN,NaN,NaN,NaN


,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
count,30998.000000,30998,30998,30998,30998,30998,30998.000000,16949.000000,16949.000000
unique,NaN,3,49,48,49,48,NaN,NaN,NaN
top,NaN,final,al,se,al,se,NaN,NaN,NaN
freq,NaN,18531,772,986,772,986,NaN,NaN,NaN
mean,2016.567359,NaN,NaN,NaN,NaN,NaN,3.974192,5.040651,5.040651
std,5.505973,NaN,NaN,NaN,NaN,NaN,5.111264,5.835317,5.835317
min,2008.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
25%,2012.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000
50%,2016.000000,NaN,NaN,NaN,NaN,NaN,2.000000,3.000000,3.000000
75%,2022.000000,NaN,NaN,NaN,NaN,NaN,7.000000,8.000000,8.000000


In [24]:
# Save Processed Datasets

contestants.to_csv(
    PROCESSED_DATA / "contestants_processed.csv",
    index=False
)

contestants_unique.to_csv(
    PROCESSED_DATA / "contestants_unique.csv",
    index=False
)

votes.to_csv(
    PROCESSED_DATA / "votes_processed.csv",
    index=False
)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.
